# Phase 13: HAR-RV Integration + Loss Optimization + Vol Trading Strategy

**Key changes from Phase 12:**
- Added 3 HAR-RV autoregressive features (rv_lag1d, rv_lag5d, rv_lag22d) -> 21 total features
- Trained three loss weight variants: 0.85/0.15, 0.50/0.50, 0.70/0.30
- Built volatility-based trading strategy backtester

**Summary of results:**
- Best vol R2: **0.8665** (vol_primary, up from 0.7719 in Phase 12)
- Best dir AUC: **0.5854** (dir_assisted, up from 0.5675 in Phase 12)
- HAR-RV benchmark: R2=0.9469 (not beaten, gap closed from 0.175 to 0.080)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

benchmarks = json.loads(open('models/phase13_benchmark_results.json').read())
backtest = json.loads(open('models/phase13_backtest_results.json').read())
backtest_returns = pd.read_csv('models/phase13_backtest_returns.csv', index_col=0, parse_dates=True)
print('Results loaded successfully')
print(f'Benchmark keys: {list(benchmarks.keys())}')
print(f'Backtest strategies: {len(backtest)}')


In [ ]:
# Three Variant Comparison - Volatility
variants = benchmarks['phase13_variants']
ref_p12 = 0.7719
ref_har = 0.9469

names = ['Phase 12\n(ref)', 'vol_primary\n(0.85/0.15)', 'balanced\n(0.50/0.50)', 'dir_assisted\n(0.70/0.30)', 'HAR-RV\n(ref)']
r2_vals = [ref_p12] + [variants[v]['vol_r2'] for v in ['vol_primary','balanced','dir_assisted']] + [ref_har]
colors = ['gray', 'steelblue', 'orange', 'green', 'red']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, r2_vals, color=colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Volatility R2')
ax.set_title('Phase 13 Variant Comparison - Volatility R2')
ax.axhline(y=ref_har, color='red', linestyle='--', alpha=0.5, label=f'HAR-RV={ref_har}')
for bar, val in zip(bars, r2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}', ha='center', fontsize=9)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig('models/phase13_vol_r2_comparison.png', dpi=150)
plt.show()

print(f"{'Variant':<25} | {'Vol R2':>8} | {'RMSE':>8} | {'Dir AUC':>8}")
print('-'*60)
print(f"{'Phase 12 (ref)':<25} | {0.7719:>8.4f} | {0.0958:>8.4f} | {0.5675:>8.4f}")
for name in ['vol_primary','balanced','dir_assisted']:
    v = variants[name]
    print(f"{name:<25} | {v['vol_r2']:>8.4f} | {v['rmse']:>8.4f} | {v['dir_auc']:>8.4f}")
print(f"{'HAR-RV (ref)':<25} | {0.9469:>8.4f} | {'N/A':>8} | {'---':>8}")


In [ ]:
# Three Variant Comparison - Direction AUC
variants = benchmarks['phase13_variants']

names = ['Phase 12\nref', 'vol_primary', 'balanced', 'dir_assisted']
auc_vals = [0.5675] + [variants[v]['dir_auc'] for v in ['vol_primary','balanced','dir_assisted']]
r2_vals = [0.7719] + [variants[v]['vol_r2'] for v in ['vol_primary','balanced','dir_assisted']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar(names, auc_vals, color=['gray','steelblue','orange','green'], edgecolor='black', alpha=0.8)
ax1.set_ylabel('Direction AUC')
ax1.set_title('Direction AUC by Variant')
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Random=0.5')
for i, v in enumerate(auc_vals):
    ax1.text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)
ax1.legend()

for i, (name, r2, auc) in enumerate(zip(['P12','0.85/0.15','0.50/0.50','0.70/0.30'], r2_vals, auc_vals)):
    ax2.scatter(r2, auc, s=120, zorder=5)
    ax2.annotate(name, (r2, auc), textcoords='offset points', xytext=(5,5), fontsize=9)
ax2.set_xlabel('Vol R2')
ax2.set_ylabel('Dir AUC')
ax2.set_title('Vol R2 vs Dir AUC Tradeoff')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('models/phase13_tradeoff.png', dpi=150)
plt.show()


## THE BENCHMARK TABLE

**Does Phase 13 beat HAR-RV R2=0.9469?**

**NO** - but Phase 13 closes the gap significantly:
- Phase 12: R2=0.7719 (gap to HAR-RV: 0.175)
- Phase 13 vol_primary: R2=0.8665 (gap to HAR-RV: 0.080)
- Gap reduced by **54%** with HAR-RV feature integration

| Model | Vol R2 | RMSE | Dir AUC |
|-------|--------|------|--------|
| Historical Average | 0.3483 | - | - |
| V2 Baseline | 0.3350 | - | - |
| HAR-RV | **0.9469** | - | - |
| Phase 12 Fusion | 0.7719 | 0.0958 | 0.5675 |
| **Phase 13 vol_primary** | **0.8665** | **0.0733** | 0.5848 |
| Phase 13 balanced | 0.7368 | 0.1029 | 0.5788 |
| Phase 13 dir_assisted | 0.8502 | 0.0777 | 0.5854 |


In [ ]:
# Gate Weight Distribution
variants = benchmarks['phase13_variants']
labels = ['Price', 'GAT', 'Document', 'Macro']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, v) in zip(axes, variants.items()):
    gates = v['gates']
    bars = ax.bar(labels, gates, color=['steelblue','orange','green','red'], edgecolor='black')
    ax.set_title(f"{name}\n(vol={v['lambda_vol']}, dir={v['lambda_dir']})")
    ax.set_ylim(0, 0.8)
    for bar, val in zip(bars, gates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.1%}', ha='center', fontsize=8)

plt.suptitle('Gate Weights by Variant', fontsize=13)
plt.tight_layout()
plt.savefig('models/phase13_gates.png', dpi=150)
plt.show()


In [ ]:
# Trading Strategy Backtest Results
bt = pd.DataFrame(backtest)
print(bt.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


In [ ]:
# Equity Curves
fig, ax = plt.subplots(figsize=(12, 6))

for col, label, color in [
    ('vol_strategy_0bp', 'Vol Strategy (0bp)', 'steelblue'),
    ('dir_strategy_0bp', 'Direction Strategy (0bp)', 'orange'),
    ('buy_and_hold', 'Buy & Hold (EW 30)', 'green'),
]:
    if col in backtest_returns.columns:
        cumret = (1 + backtest_returns[col].fillna(0)).cumprod()
        ax.plot(cumret.index, cumret.values, label=label, color=color, linewidth=1.5)

ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
ax.set_title('Phase 13 Strategy Equity Curves (Test Period)')
ax.set_ylabel('Cumulative Return')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('models/phase13_equity_curves.png', dpi=150)
plt.show()


## Phase 13 Summary

### Volatility Forecasting
- **HAR-RV features** (rv_lag1d, rv_lag5d, rv_lag22d) improved vol R2 from 0.7719 to 0.8665 (+12.2%)
- Gap to HAR-RV benchmark closed from 0.175 to 0.080 (54% reduction)
- vol_primary (0.85/0.15) is the best variant for volatility prediction
- dir_assisted (0.70/0.30) is best for direction while maintaining strong vol R2=0.8502

### Loss Weight Optimization
- 0.85/0.15 (vol_primary): Best vol R2=0.8665, Dir AUC=0.5848
- 0.70/0.30 (dir_assisted): Vol R2=0.8502, Best Dir AUC=0.5854
- 0.50/0.50 (balanced): Worst on both metrics - too much direction weight harms vol learning

### Trading Strategy
- Buy & Hold dominates in a strong bull market (2024-2025): Sharpe=0.652
- Direction strategy outperforms vol strategy: Sharpe=0.604 vs -0.453
- Vol strategy underperforms - vol mispricing signals are noisy in trending markets
- Market-neutral strategies lose to directional exposure in bull markets

### Key Files
- `models/phase13_fusion_vol_primary.pt` - best vol model
- `models/phase13_fusion_dir_assisted.pt` - best direction model
- `models/phase13_benchmark_results.json` - full benchmark comparison
- `models/phase13_backtest_results.json` - trading strategy results
